Q&A chatbot with chat history

In [ ]:
from openai import OpenAI
import gradio as gr

OLLAMA_BASE_URL = "http://localhost:11434/v1"
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

system_prompt = """
You are a professional software coding master. 
Explain code in a simple beginner-friendly way.
Give examples and suggest better alternatives if possible.
"""

def chat_fn(message, history, model):
    history = history or []

    
    messages = [{"role": "system", "content": system_prompt}]

    for user_msg, bot_msg in history:
        messages.append({"role": "user", "content": user_msg})
        messages.append({"role": "assistant", "content": bot_msg})

    messages.append({"role": "user", "content": message})

   
    stream = ollama.chat.completions.create(
        model=model,
        messages=messages,
        stream=True
    )

    reply = ""
    for chunk in stream:
        reply += chunk.choices[0].delta.content or ""
        yield history + [(message, reply)], history + [(message, reply)]



with gr.Blocks() as app:
    gr.Markdown("# 💻 AI Code Assistant")

    model = gr.Dropdown(
        ["llama3.2:1b", "phi"],
        value="llama3.2:1b",
        label="Choose Model"
    )

    chatbot = gr.Chatbot()

    msg = gr.Textbox(
        placeholder="Ask a question or paste code...",
        label="Your Input"
    )

    send_btn = gr.Button("Send")
    clear_btn = gr.Button("Clear Chat")

    state = gr.State([])

    
    send_btn.click(
        chat_fn,
        inputs=[msg, state, model],
        outputs=[chatbot, state]
    )

    
    clear_btn.click(
        lambda: ([], []),
        inputs=[],
        outputs=[chatbot, state]
    )

app.launch()